In [28]:
import pandas as pd
import tensorflow as tf

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

from sklearn.utils import shuffle
from sklearn import preprocessing
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

Instructions for updating:
non-resource variables are not supported in the long term


### Reading the dataset

In [37]:
def label_encode(label):
	val = []
	if label == "Iris-setosa":
		val = [1,0,0]
	elif label == "Iris-versicolor":
		val = [0,1,0]
	elif label == "Iris-virginica":
		val = [0,0,1]
	return val

def data_encode(file):
	X = []
	Y = []
	train_file = open(file, 'r')
	for line in train_file.read().strip().split('\n'):
		line = line.split(',')
		X.append([line[0],line[1],line[2],line[3]])
		Y.append(label_encode(line[4]))
	return X,Y

In [38]:
X,y = data_encode("IRIS.csv")
l = int(len(df)/3)
X_train = X[0:l]
y_train = y[0:l]
X_val = X[l:]
y_val = y[l:]

In [ ]:
#parametros
learning_rate = 0.01
training_epochs = 5000
display_steps = 100
num_features = len(X[0])

n_input = 4
n_hidden = 10
n_output = 3


input_X = tf.placeholder('float32',shape =(None,num_features))
input_Y = tf.placeholder('float32',shape = (None,n_output))

# X = tf.compat.v1.placeholder("float",[None,n_input])
# Y = tf.compat.v1.placeholder("float",[None,n_output])

weights = {"hidden": tf.Variable(tf.random.normal([n_input,n_hidden])),
    "output": tf.Variable(tf.random.normal([n_hidden,n_output]))}

bias = {"hidden": tf.Variable(tf.random.normal([n_hidden])),
    "output": tf.Variable(tf.random.normal([n_output]))}

def model(X, weights, bias):
    layer1 = tf.add(tf.matmul(X, weights["hidden"]),bias["hidden"])
    layer1 = tf.nn.relu(layer1)

    output_layer = tf.matmul(layer1,weights["output"]) + bias["output"]
    return output_layer


pred = model(input_X,weights,bias)

cost = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(logits=pred, labels=input_Y))
optimizador = tf.train.AdamOptimizer(learning_rate).minimize(cost)

init = tf.global_variables_initializer()

with tf.Session() as sess:
    sess.run(init)

    for epochs in range(training_epochs):
        _, c= sess.run([optimizador,cost],feed_dict = {X: train_X, Y: train_Y})
        if(epochs + 1) % display_steps == 0:
            print("Epoch:",epochs+1,"Cost:", c)
    print("Optimization Finished")

    test_result = sess.run(pred,feed_dict = {X: train_X})
    correct_prediction = tf.equal(tf.argmax(test_result,1),tf.argmax(train_Y,1))
    accuracy = tf.reduce_mean(tf.cast(correct_prediction,"float"))

    print("accuracy:", accuracy.eval({X: test_X, Y: test_Y}))

Instructions for updating:

Future major versions of TensorFlow will allow gradients to flow
into the labels input on backprop by default.

See `tf.nn.softmax_cross_entropy_with_logits_v2`.

